# Chương 10: Thực hành - Thị giác máy tính (Computer Vision & CNNs)

**Mục tiêu bài thực hành:**
1. **Phép toán Tích chập 2D từ đầu:** Cài đặt hàm `conv2d` với Stride và Padding; áp dụng các bộ lọc truyền thống (Sobel, Gaussian, Sharpen) lên hình ảnh.
2. **Khối phần dư ResNet (Residual Block):** Triển khai cơ chế Skip Connection $\mathbf{y} = \mathcal{F}(\mathbf{x}) + \mathbf{x}$ chỉ bằng `NumPy`.
3. **Trực quan hóa Feature Maps:** Quan sát cách các tầng tích chập trích xuất đặc trưng biên cạnh và hình thái.
4. **Cơ chế Nhận diện đối tượng:** Tự cài đặt hàm tính chỉ số $\text{IoU}$ (Intersection over Union) và thuật toán lọc hộp trùng lặp **Non-Maximum Suppression (NMS)**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cố định hạt giống ngẫu nhiên
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Cài đặt Phép toán Tích chập 2D (Conv2D) và Các Bộ lọc Truyền thống

Công thức kích thước đầu ra: $H_{\text{out}} = \lfloor \frac{H - K + 2P}{S} \rfloor + 1$.

In [ ]:
def conv2d_scratch(img, kernel, stride=1, padding=1):
    H, W = img.shape
    Kh, Kw = kernel.shape
    
    # Thêm đệm Zero-padding
    img_padded = np.pad(img, ((padding, padding), (padding, padding)), mode='constant', constant_values=0)
    
    H_out = (H - Kh + 2 * padding) // stride + 1
    W_out = (W - Kw + 2 * padding) // stride + 1
    out = np.zeros((H_out, W_out))
    
    for i in range(H_out):
        for j in range(W_out):
            r_start = i * stride
            c_start = j * stride
            window = img_padded[r_start:r_start+Kh, c_start:c_start+Kw]
            out[i, j] = np.sum(window * kernel)
            
    return out

# Tạo một ảnh mẫu 2D có hình chữ nhật và hình tròn
synthetic_img = np.zeros((64, 64))
synthetic_img[10:35, 10:35] = 1.0 # Hình vuông trắng
y_idx, x_idx = np.ogrid[:64, :64]
circle_mask = (x_idx - 45)**2 + (y_idx - 45)**2 <= 12**2
synthetic_img[circle_mask] = 0.8 # Hình tròn trắng

# Các kernel bộ lọc kinh điển
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])   # Biên dọc
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])   # Biên ngang
gaussian_blur = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16.0 # Làm mờ
sharpen = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])  # Làm sắc nét

edge_x = conv2d_scratch(synthetic_img, sobel_x)
edge_y = conv2d_scratch(synthetic_img, sobel_y)
blurred = conv2d_scratch(synthetic_img, gaussian_blur)
sharpened = conv2d_scratch(synthetic_img, sharpen)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
imgs = [
    ("Ảnh gốc ban đầu", synthetic_img, 'gray'),
    ("Biên dọc (Sobel X)", edge_x, 'magma'),
    ("Biên ngang (Sobel Y)", edge_y, 'magma'),
    ("Làm mờ (Gaussian Blur)", blurred, 'gray'),
    ("Làm sắc nét (Sharpen)", sharpened, 'gray')
]

for idx, (title, im_data, colormap) in enumerate(imgs):
    axes[idx].imshow(im_data, cmap=colormap)
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle("Tác động của Các Bộ lọc Không gian Tích chập (2D Convolution Filters)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Khối phần dư ResNet (Residual Block with Skip Connection)

Triển khai công thức: $\mathbf{y} = \text{ReLU}(\text{Conv}_2(\text{ReLU}(\text{Conv}_1(\mathbf{x}))) + \mathbf{x})$.

In [ ]:
class ResidualBlock:
    def __init__(self, size=3):
        # Khởi tạo hai bộ lọc ngẫu nhiên 3x3
        self.k1 = np.random.randn(size, size) * 0.1
        self.k2 = np.random.randn(size, size) * 0.1
        
    def forward(self, x):
        # Tầng tích chập 1 + ReLU
        z1 = conv2d_scratch(x, self.k1, padding=1)
        a1 = np.maximum(0, z1)
        
        # Tầng tích chập 2
        z2 = conv2d_scratch(a1, self.k2, padding=1)
        
        # Skip Connection: Cộng trực tiếp x vào kết quả
        out = np.maximum(0, z2 + x)
        return out

res_block = ResidualBlock()
res_output = res_block.forward(synthetic_img)
print(f"Kích thước ảnh đầu vào: {synthetic_img.shape} -> Kích thước đầu ra khối ResNet: {res_output.shape}")

## 3. Nhận diện Đối tượng: Tính IoU và Thuật toán Non-Maximum Suppression (NMS)

- $\text{IoU} = \frac{\text{Diện tích Giao nhau (Intersection)}}{\text{Diện tích Hợp nhất (Union)}}$
- NMS loại bỏ các hộp bao dư thừa có điểm IoU cao với hộp có độ tin cậy cao nhất.

In [ ]:
def compute_iou(box1, box2):
    """
    box: [x1, y1, x2, y2] (tọa độ góc trên-trái và dưới-phải)
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    intersection = inter_w * inter_h
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    if union == 0:
        return 0.0
    return intersection / union

def non_maximum_suppression(boxes, scores, iou_thresh=0.4):
    order = np.argsort(scores)[::-1] # Sắp xếp giảm dần theo điểm tin cậy
    keep = []
    
    while len(order) > 0:
        idx = order[0]
        keep.append(idx)
        if len(order) == 1:
            break
            
        # Tính IoU giữa hộp tốt nhất với các hộp còn lại
        rest = order[1:]
        ious = np.array([compute_iou(boxes[idx], boxes[j]) for j in rest])
        # Giữ lại các hộp có IoU <= ngưỡng
        mask = ious <= iou_thresh
        order = rest[mask]
        
    return keep

# Mô phỏng 5 hộp bao dự đoán của detector cho 1 đối tượng duy nhất kèm nhiễu
boxes_pred = np.array([
    [12, 12, 34, 34], # Box 1 (Đúng nhất)
    [10, 11, 35, 36], # Box 2 (Trùng lặp nhiều)
    [14, 13, 32, 33], # Box 3 (Trùng lặp nhiều)
    [42, 42, 58, 58], # Box 4 (Đối tượng khác ở góc)
    [40, 41, 57, 56], # Box 5 (Trùng với Box 4)
])
scores_pred = np.array([0.95, 0.82, 0.76, 0.90, 0.68])

keep_indices = non_maximum_suppression(boxes_pred, scores_pred, iou_thresh=0.4)
print(f"Các hộp bao được giữ lại sau NMS: {keep_indices} (Điểm tin cậy tương ứng: {scores_pred[keep_indices]})")

# Trực quan hóa
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(synthetic_img, cmap='gray')
for b in boxes_pred:
    rect = plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, edgecolor='yellow', linewidth=2)
    ax1.add_patch(rect)
ax1.set_title("Trước NMS: 5 Hộp Dự đoán Bị Trùng lặp", fontsize=11, fontweight='bold')
ax1.axis('off')

ax2.imshow(synthetic_img, cmap='gray')
for idx in keep_indices:
    b = boxes_pred[idx]
    rect = plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, edgecolor='lime', linewidth=2.5)
    ax2.add_patch(rect)
ax2.set_title("Sau NMS: Chỉ giữ lại 2 Hộp Tối ưu Tuyệt đối", fontsize=11, fontweight='bold')
ax2.axis('off')

plt.suptitle("Cơ chế Lọc Hộp bao Trùng lặp Non-Maximum Suppression (NMS)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Tổng kết bài học thực hành

1. **Tích chập 2D:** Biến đổi cục bộ hiệu quả; các bộ lọc toán học kinh điển (Sobel, Gaussian) là tiền thân của các kernel tự học trong CNN.
2. **ResNet:** Cơ chế Skip Connection bảo toàn thông tin gốc và tạo đường dẫn gradient thẳng, cho phép huấn luyện mạng nơ-ron sâu hàng trăm tầng mà không bị suy thoái.
3. **IoU & NMS:** Hai công cụ nền tảng xử lý hậu kỳ bắt buộc của mọi hệ thống nhận diện đối tượng hiện đại (YOLO, Faster R-CNN) để loại bỏ các hộp bao dư thừa.